In [2]:
!pip install ultralytics opencv-python matplotlib pillow yolo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.5/122.5 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.

Loading the dataset I stored and uploaded my image datasets of folder_2 in Google Drive as it allows me to connect my Google Drive directly to my Colab environment. The reason is because this approach emulates a production workflow where datasets are stored in a cloud repository rather than on the local machine. This method is the same one I used when working with large image datasets that would otherwise be impractical to upload repeatedly or store locally. It also supports scalability where as datasets grow in size, they can still be accessed without changing the code structure

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
from ultralytics import YOLO
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

# Paths for images and models in Drive
folder_1_path = "/content/drive/MyDrive/Questions_RND/Question2/cones/folder_1"  # pathway of image datasets
model_path = "/content/drive/MyDrive/Questions_RND/Question2/cones/Model/Cone.pt" #pathway of yolov8 model
output_path="/content/drive/MyDrive/Questions_RND/Question2/cones//output"
# Verify the file exists
if not os.path.exists(model_path):
    raise FileNotFoundError(f"Model not found at: {model_path}")

# Load YOLOv8 model
model = YOLO(model_path)



Here is the general outline of steps I take to answer question 2.1 .
1. Libraries and implementation
I started by installing the Ultralytics package and importing essential libraries including the YOLOV8 model, Python Image Liv for image processing, Matplotlib for visualization, and os for file operation.

2. Loading the YOLOv8 model
I specified the path to the pre-trained weights file (Cone.pt) given by assessment and loaded it using YOLO(model_path).  

3. Procesing images
For each image in the specified folder, I first filtered for common image formats (JPG,PNG) then processed them sequentially. This approach was important to handle multiple images efficiently while managing memory constraints. Since there are a total of 6 images only in folder_1 ,I am able to process those images individually rather than batching them.

4. Running object detection
For each image, I loaded it using python image lib and passed it to the YOLOV8 model prediction function. This detection phase identifies all cones present in the image through bounding box prediction
For every detected cone, I calculated the geometric center point of its bounding box by averaging the x-coordinates and y-coordinates of opposite corners. The formula and approch was this are taken from stachexchange. This was done to calculate the approximate centre location for each cone that's less sensitive to bounding box variations than corners.

5. Drawing the line between objects
Using Python image lib drawing functions, I connected consecutive sorted centers with thick red lines.The red lines create clear visible connections between cones

6. Displaying and saving results
 I displayed each processed image using Matplotlib with axes hidden for cleaner presentation, and saved a copy  with "LineDrawn_". This dual-output is for 2 reasons: First being the immediate real-time verification during development, Second being it save and  preserves result for assessment submission.

In [1]:
import os
from ultralytics import YOLO
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

# Verifying if the file exists
if not os.path.exists(model_path):
    raise FileNotFoundError(f"Model not found at: {model_path}")

# Loading the yolov8 model
model = YOLO(model_path)
print("Model loaded successfully!")

def process_folder1_images(folder_path):
    # Get all image files
    image_files = [f for f in os.listdir(folder_path) if f.endswith(('.jpg', '.png'))]
    print(f"Found {len(image_files)} images in folder")

    for img_file in image_files:
        img_path = os.path.join(folder_path, img_file)
        print(f"\nProcessing: {img_file}")

        # Open and verify image infos
        img = Image.open(img_path)
        print(f"Image size: {img.size}")

        # Detecting the objects in images
        results = model(img)
        print(f"{len(results[0].boxes)} cones are detected")

        # Extract cone centers
        centers = []
        for box in results[0].boxes.xyxy:
            x1, y1, x2, y2 = box.tolist()
            center_x = int((x1 + x2) / 2)
            center_y = int((y1 + y2) / 2)
            centers.append((center_x, center_y))
            print(f"  Cone at: ({center_x}, {center_y})")

        # Drawing connections between centers of object
        draw = ImageDraw.Draw(img)

        if len(centers) > 1:
            centers.sort(key=lambda x: x[0])  # Sort centers from left to right

            for i in range(1, len(centers)):
                draw.line([centers[i-1], centers[i]], fill="red", width=3)  # Drawing the connecting lines
        else:
            print("Not enough cones to draw lines")

        # Display results
        plt.figure(figsize=(11, 6))
        plt.imshow(img)
        plt.title(f"Cone Layout: {img_file}")
        plt.axis('off')
        plt.show()

        # Save results
        output = os.path.join(output_path, "LineDrawn_" + img_file)
        img.save(output)
        print(f"Saved processed image to: {output}")

process_folder1_images(folder_1_path)


ModuleNotFoundError: No module named 'ultralytics'